# 1.2 逐元素算子与 set_vec_tile_shapes

上一节用一个最小 kernel 建立了 PyPTO 算子的基本闭环：host 侧准备真实 Tensor，kernel 侧描述计算，结果写入 `out`，最后用 PyTorch 参考实现验证。本节进入第一类具体计算：逐元素算子。

逐元素算子适合作为第一个计算类型来学习，因为它的 shape 变化最直观：大多数情况下，输入和输出 shape 保持一致；如果输入 shape 不同，也要先通过广播规则对齐。理解这一点后，`pypto.add`、`pypto.mul`、`pypto.maximum`、`pypto.exp` 等 API 都可以按同一种方式阅读。

本节从三个层次展开：先理解“每个位置各算各的”，再看标量和广播如何参与逐元素计算，最后用加法和融合乘加 ReLU 两个 kernel 完成验证闭环。

---
## 前置说明

本节在第一个代码单元格中集中放置环境准备代码（import、设备选择、`RUN_MODE` 等）。后续每个可运行模块在定义 kernel 前只调用 `reset_pypto_notebook_state()` 重置编译状态，不再重复完整的环境初始化。

环境准备的具体含义参见 `01.01_chapter_intro.ipynb` §1。

In [ ]:
import os

# 环境准备：参见 01.01_chapter_intro.ipynb §1 的详细说明
# 如需指定其他 NPU，请在启动 Notebook 前设置 ASCEND_RT_VISIBLE_DEVICES。
# os.environ.setdefault("ASCEND_RT_VISIBLE_DEVICES", "13")

import torch
import pypto
from numpy.testing import assert_allclose

try:
    import torch_npu  # noqa: F401
except Exception:
    torch_npu = None


def reset_pypto_notebook_state():
    "Clean stale PyPTO recording state before defining a JIT kernel."
    try:
        pypto.reset()
    except Exception:
        pass

    try:
        from pypto._controller import Controller
        Controller.end_function()
    except Exception:
        pass


def get_device():
    if torch_npu is None:
        return "cpu"

    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    try:
        torch.npu.set_device(device_id)
    except Exception as exc:
        print("NPU device is not ready:", exc)
        return "cpu"
    return f"npu:{device_id}"


reset_pypto_notebook_state()
device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

# print("ASCEND_RT_VISIBLE_DEVICES:", os.environ.get("ASCEND_RT_VISIBLE_DEVICES", "<未设置>"))
print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<未设置，默认 0>"))
print("device:", device)


ASCEND_RT_VISIBLE_DEVICES: 13
TILE_FWK_DEVICE_ID: <未设置，默认 0>
device: npu:0


## 2. 什么是逐元素计算

逐元素计算可以直观理解为“每个位置各算各的”。例如输入 `x` 和 `scale` 的形状都是 `[8, 8]`，表达式：

```python
y = x * scale
```

表示输出中 `(i, j)` 位置的值只依赖 `x[i, j]` 和 `scale[i, j]`：

```text
y[i, j] = x[i, j] * scale[i, j]
```

不同位置之间没有相互依赖，也不会把一组元素合成一个元素。这个特点和后面要学习的规约不同：逐元素计算通常保持 shape，规约通常会压缩某个维度。

常见逐元素计算包括：

| 类型 | 示例 |
| --- | --- |
| 算术运算 | `x + y`、`x * y`、`x / y` |
| 标量混合运算 | `x * 2.0`、`x + 1.0` |
| 激活函数 | `maximum(x, 0)`、`sigmoid(x)` |
| 数学函数 | `exp(x)`、`sqrt(x)`、`rsqrt(x)` |

因为这类计算天然适合按元素并行处理，所以在 PyPTO 中通常使用 `pypto.set_vec_tile_shapes` 设置向量计算的 Tile Shape。

## 2.1 逐元素算子的常见模式

逐元素算子不只包含最基础的加法。实际使用时，它通常会以几种固定模式出现：两个 Tensor 对应位置计算、Tensor 和标量计算、带缩放参数的计算、不同 shape 之间的广播计算，以及对每个元素独立应用数学函数。

这些模式可以这样归纳：

| 模式 | 代表表达式 | 数学含义 | 学习重点 |
| --- | --- | --- | --- |
| 同 shape 二元计算 | `pypto.add(a, b)`、`pypto.mul(a, b)` | 两个 Tensor 对应位置逐元素计算 | 输出 shape 通常与输入一致 |
| Tensor 与标量计算 | `pypto.add(x, scalar)`、`pypto.mul(x, scalar)` | 同一个数字作用到 Tensor 的每个元素 | 标量不会改变输出 shape |
| 带缩放参数计算 | `pypto.add(a, b, alpha=alpha)` | `a + alpha * b` | 先缩放第二个输入，再逐元素相加 |
| 广播计算 | `pypto.add(x, bias)` | 较小 Tensor 按广播规则参与计算 | 先判断 shape 是否能右对齐 |
| 单输入数学函数 | `pypto.exp(x)`、`pypto.log(x)`、`pypto.sqrt(x)`、`pypto.rsqrt(x)` | 对每个元素独立应用函数 | 一个输入位置得到一个输出位置 |
| 限幅与取整 | `pypto.clip(x, min, max)`、`pypto.round(x)`、`pypto.ceil(x)`、`pypto.floor(x)` | 每个元素独立限制或取整 | 数值变换不改变位置关系 |
| 符号与幂运算 | `pypto.neg(x)`、`pypto.pow(a, b)` | 取负或逐元素求幂 | 仍然属于逐元素模式 |

这些写法的 kernel 结构基本一致：

```python
@pypto.frontend.jit(...)
def xxx_kernel(...):
    pypto.set_vec_tile_shapes(...)
    out[:] = pypto.xxx(...)
```

有些示例会写成 `out.move(...)`。在本节语境下，两种写法都表达同一个核心动作：把右侧 PyPTO 表达式的结果写入调用者传入的输出 Tensor。

因此，学习逐元素算子时不需要先背完整 API 列表。更重要的是先识别三个问题：

1. 这个算子是不是每个输出位置独立计算？
2. 输入 shape 是完全相同，还是需要广播？
3. 结果是如何写回 `out` 的？

## 3. set_vec_tile_shapes 的作用

在正式写代码之前，需要先澄清一个容易误解的点：`set_vec_tile_shapes` 不是数学公式的一部分。

例如：

```python
pypto.set_vec_tile_shapes(8, 8)
```

可以先理解为：当处理二维 Tensor 时，编译系统可以按 `8 x 8` 的小块组织向量计算。这个设置影响的是数据如何分块、搬运和调度，不改变 `out = x + y` 的数学结果。

换句话说：

- `out = x + y` 决定“算什么”。
- `set_vec_tile_shapes(8, 8)` 影响“怎么组织执行”。

因此，同一个逐元素公式可以搭配不同的 vec tile 配置；只要配置合法，数学结果应该一致，差别主要体现在执行组织和性能上。
在 Notebook 中，每个可运行模块都先清理 PyPTO 的记录状态，再定义当前模块的 JIT kernel。这样一个模块的失败不会影响下一个模块的验证。


## 3.1 标量运算：Tensor 和单个数字一起计算

逐元素计算不仅可以发生在两个 Tensor 之间，也可以发生在 Tensor 和一个标量之间。标量就是单个数字，例如 `2.0`、`0.5`。

如果 `x` 的 shape 是 `[8, 8]`，那么：

```python
y = x * 2.0
```

表示 `x` 中每个位置都乘以同一个数字 `2.0`，输出 shape 仍然是 `[8, 8]`。这类写法在模型中很常见，例如缩放、加偏移、归一化等。


In [9]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def scalar_mul_add_kernel(
    x: pypto.Tensor([], pypto.DT_FP16),
    out: pypto.Tensor([], pypto.DT_FP16)):
    pypto.set_vec_tile_shapes(8, 8)
    out.move(pypto.add(pypto.mul(x, 2.0), 1.0))


def main_scalar_mul_add():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.randn((8, 8), dtype=torch.float16, device=device)
    out = torch.empty_like(x)

    scalar_mul_add_kernel(x, out)
    ref = x * 2.0 + 1.0

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-2, atol=1e-2)
    print("scalar_mul_add_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("输入 shape:", tuple(x.shape), "输出 shape:", tuple(out.shape))
    print("最大误差:", max_diff)
    print("输入前 2x4:", x[:2, :4].cpu())
    print("输出前 2x4:", out[:2, :4].cpu())


main_scalar_mul_add()


scalar_mul_add_kernel 验证通过
device: npu:0 run_mode: 0
输入 shape: (8, 8) 输出 shape: (8, 8)
最大误差: 0.0
输入前 2x4: tensor([[-0.0064,  1.5098, -0.9702,  0.4934],
        [ 0.5337, -1.0098,  0.2935, -0.2979]], dtype=torch.float16)
输出前 2x4: tensor([[ 0.9873,  4.0195, -0.9404,  1.9863],
        [ 2.0664, -1.0195,  1.5869,  0.4043]], dtype=torch.float16)


`scalar_mul_add_kernel` 演示 Tensor 和标量一起做逐元素计算。标量 `2.0`、`1.0` 会自动应用到输入 Tensor 的每个位置，输出 shape 不会因此改变。


上面的算子等价于普通 PyTorch 表达式：

```python
out = x * 2.0 + 1.0
```

这里 `2.0` 和 `1.0` 都是标量。PyPTO 会把它们应用到 `x` 的每个元素上。


这个单元用来验证 `scalar_mul_add_kernel`。它先在当前设备上构造一组小尺寸输入，再提前分配输出 Tensor 并调用 PyPTO kernel。随后用普通 PyTorch 写出等价的 PyTorch 参考结果，打印 shape、样例值和最大误差；最后的 `assert_close` 是正式检查，误差超出阈值时会直接报错。


**预期输出说明**

运行成功后，会看到 `scalar_mul_add_kernel 验证通过`，并打印输入 shape、输出 shape、最大误差和输出前 `2x4` 的数值。


## 3.2 广播机制：让较小的 Tensor 自动扩展参与计算

广播机制用于处理 shape 不完全相同但可以对齐的 Tensor。理解广播前，先明确一个基础点：`shape` 描述的是每个维度的长度，不是 Tensor 里的具体数值。

```python
torch.tensor([1, 2]).shape
```

结果是 `[2]`，表示这是一个一维 Tensor，长度为 2。它不是 `[1, 2]`，因为 `[1, 2]` 是二维 shape，需要多一层中括号：

```python
torch.tensor([[1, 2]]).shape  # [1, 2]
```

广播从右往左对齐 shape。假设 `x` 的 shape 是 `[4, 3]`，`bias` 的 shape 是 `[3]`：

```python
x = [[x00, x01, x02],
     [x10, x11, x12],
     [x20, x21, x22],
     [x30, x31, x32]]

bias = [b0, b1, b2]
```

右对齐后，可以把 `bias` 看成 `[1, 3]`，再广播到 `[4, 3]`。计算 `x + bias` 时，相当于：

```python
[[x00 + b0, x01 + b1, x02 + b2],
 [x10 + b0, x11 + b1, x12 + b2],
 [x20 + b0, x21 + b1, x22 + b2],
 [x30 + b0, x31 + b1, x32 + b2]]
```

广播不会真的要求复制很多份数据，但计算语义上可以这样理解。逐元素计算强调“每个输出位置独立计算”，广播强调“输入 shape 不同时如何对齐到共同输出 shape”。

In [10]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def broadcast_bias_kernel(
    x: pypto.Tensor([], pypto.DT_FP16),
    bias: pypto.Tensor([], pypto.DT_FP16),
    out: pypto.Tensor([], pypto.DT_FP16)):
    pypto.set_vec_tile_shapes(8, 8)
    out.move(pypto.add(x, bias))


def main_broadcast_bias():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.randn((8, 8), dtype=torch.float16, device=device)
    bias = torch.randn((8,), dtype=torch.float16, device=device)
    out = torch.empty_like(x)

    broadcast_bias_kernel(x, bias, out)
    ref = x + bias

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-2, atol=1e-2)
    print("broadcast_bias_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("x shape:", tuple(x.shape), "bias shape:", tuple(bias.shape), "输出 shape:", tuple(out.shape))
    print("最大误差:", max_diff)
    print("bias 前 4 个:", bias[:4].cpu())
    print("输出前 2x4:", out[:2, :4].cpu())


main_broadcast_bias()


broadcast_bias_kernel 验证通过
device: npu:0 run_mode: 0
x shape: (8, 8) bias shape: (8,) 输出 shape: (8, 8)
最大误差: 0.0
bias 前 4 个: tensor([-2.5586, -1.8564,  0.4172,  1.0781], dtype=torch.float16)
输出前 2x4: tensor([[-4.0781, -1.9531, -0.8467,  0.6338],
        [-3.2891, -2.5391,  0.5449,  0.8384]], dtype=torch.float16)


`broadcast_bias_kernel` 展示广播机制。较小的 `bias` Tensor 会沿缺失维度扩展到和 `x` 兼容的形状，然后参与逐元素加法；广播只影响对齐方式，不改变加法的数学含义。


这个单元用来验证 `broadcast_bias_kernel`。它先在当前设备上构造一组小尺寸输入，再提前分配输出 Tensor 并调用 PyPTO kernel。随后用普通 PyTorch 写出等价的 PyTorch 参考结果，打印 shape、样例值和最大误差；最后的 `assert_close` 是正式检查，误差超出阈值时会直接报错。


**预期输出说明**

运行成功后，会看到 `broadcast_bias_kernel 验证通过`，并打印 `x`、`bias`、输出的 shape。这里可以观察到 `[8]` 的 `bias` 如何参与 `[8, 8]` 的广播计算。


## 4. 从最小逐元素加法开始

下面先实现一个加法算子。虽然功能很基础，但它展示了逐元素算子的基本写法。


## 4.1 加法算子的输入输出规格

在编写代码前，先明确这个最小算子的规格。Ascend C 教程中经常先做“算子分析”，PyPTO 教程也应保持这个习惯。

| 项目 | 说明 |
| --- | --- |
| 数学表达式 | `out = x + y` |
| 输入 `x` | FP16 Tensor，示例 shape 为 `[8, 8]` |
| 输入 `y` | FP16 Tensor，shape 与 `x` 相同 |
| 输出 `out` | FP16 Tensor，shape 与 `x` 相同 |
| 计算类型 | 逐元素向量计算 |
| Tile 设置 | `pypto.set_vec_tile_shapes(8, 8)` |

这里的重点不是加法本身，而是通过一个足够简单的例子看清 PyPTO 算子的基本写法。


In [11]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def add_kernel(
    x: pypto.Tensor([], pypto.DT_FP16),
    y: pypto.Tensor([], pypto.DT_FP16),
    out: pypto.Tensor([], pypto.DT_FP16)):
    pypto.set_vec_tile_shapes(8, 8)
    out.move(pypto.add(x, y))


def main_add():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.randn((8, 8), dtype=torch.float16, device=device)
    y = torch.randn((8, 8), dtype=torch.float16, device=device)
    out = torch.empty_like(x)

    add_kernel(x, y, out)
    ref = x + y

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-2, atol=1e-2)
    print("add_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("x shape:", tuple(x.shape), "y shape:", tuple(y.shape), "输出 shape:", tuple(out.shape))
    print("最大误差:", max_diff)
    print("输出前 2x4:", out[:2, :4].cpu())


main_add()


add_kernel 验证通过
device: npu:0 run_mode: 0
x shape: (8, 8) y shape: (8, 8) 输出 shape: (8, 8)
最大误差: 0.0
输出前 2x4: tensor([[ 0.2217, -1.7070,  0.4697,  2.4961],
        [-0.9346, -2.1191, -1.2070,  0.3237]], dtype=torch.float16)


`add_kernel` 只做一步逐元素加法。`pypto.add(x, y)` 描述计算表达式，`out.move(...)` 把结果写入调用者传入的输出 Tensor；Tile Shape 只影响底层分块执行，不改变 `out = x + y` 的数学含义。

### 代码细节解释

这段代码可以分成三层理解：

1. **函数签名**：`x`、`y`、`out` 都被标注为 `pypto.Tensor([], pypto.DT_FP16)`，表示它们是进入 PyPTO 编译流程的 Tensor 参数；真实数据会在调用 kernel 时传入。
2. **Tile 设置**：`set_vec_tile_shapes(8, 8)` 表示这个二维向量计算按 `8 x 8` Tile 组织。它不改变 `x + y` 的数学结果。
3. **输出写回**：`out.move(pypto.add(x, y))` 将加法结果写回输出 Tensor。没有这一步，调用者提前分配的 `out` 不会得到最终结果。

用自然语言描述就是：接收两个 FP16 Tensor，按 vec tile 组织逐元素加法，然后把结果写入 `out`。

这段代码中，`pypto.add(x, y)` 表达逐元素加法，`out.move(...)` 将结果写回输出 Tensor。为了验证它是否正确，我们使用 PyTorch 的 `x + y` 作为参考实现。


这个单元用来验证 `add_kernel`。它先在当前设备上构造一组小尺寸输入，再提前分配输出 Tensor 并调用 PyPTO kernel。随后用普通 PyTorch 写出等价的 PyTorch 参考结果，打印 shape、样例值和最大误差；最后的 `assert_close` 是正式检查，误差超出阈值时会直接报错。


**预期输出说明**

运行成功后，会看到 `add_kernel 验证通过`，并打印输入/输出 shape、最大误差和输出前几项。


## 5. 融合乘加与 ReLU

接下来把示例扩展为一个更贴近实际模型的融合算子：

```python
y = x * scale + bias
out = maximum(y, 0)
```

如果使用普通 PyTorch，这段计算可能由多个算子串联完成。PyPTO 的优势在于可以在一个 JIT 函数中描述完整计算，让编译流程看到完整的数据依赖。


## 5.1 融合算子的输入输出规格

融合乘加 ReLU 相比加法多了两个输入和一个激活步骤。先明确规格，再看代码会更容易理解。

| 项目 | 说明 |
| --- | --- |
| 数学表达式 | `out = maximum(x * scale + bias, 0)` |
| 输入 `x` | FP16 Tensor，示例 shape 为 `[8, 8]` |
| 输入 `scale` | FP16 Tensor，shape 与 `x` 相同 |
| 输入 `bias` | FP16 Tensor，shape 与 `x` 相同 |
| 输出 `out` | FP16 Tensor，shape 与 `x` 相同 |
| 中间变量 `y` | 保存 `x * scale + bias` 的表达式结果 |
| 计算类型 | 多个逐元素计算组合 |

这个示例说明：复杂一点的逐元素算子通常不是新的神秘能力，而是基础逐元素操作的组合。


In [12]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def fused_mul_add_relu_kernel(
    x: pypto.Tensor([], pypto.DT_FP16),
    scale: pypto.Tensor([], pypto.DT_FP16),
    bias: pypto.Tensor([], pypto.DT_FP16),
    out: pypto.Tensor([], pypto.DT_FP16)):
    pypto.set_vec_tile_shapes(8, 8)
    y = pypto.add(pypto.mul(x, scale), bias)
    out.move(pypto.maximum(y, 0.0))


def main_fused_mul_add_relu():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.randn((8, 8), dtype=torch.float16, device=device)
    scale = torch.randn((8, 8), dtype=torch.float16, device=device)
    bias = torch.randn((8, 8), dtype=torch.float16, device=device)
    out = torch.empty_like(x)

    fused_mul_add_relu_kernel(x, scale, bias, out)
    zero = torch.tensor(0.0, dtype=torch.float16, device=device)
    ref = torch.maximum(x * scale + bias, zero)

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-2, atol=1e-2)
    print("fused_mul_add_relu_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("x shape:", tuple(x.shape), "scale shape:", tuple(scale.shape), "bias shape:", tuple(bias.shape))
    print("输出 shape:", tuple(out.shape), "最大误差:", max_diff)
    print("输出前 2x4:", out[:2, :4].cpu())


main_fused_mul_add_relu()


fused_mul_add_relu_kernel 验证通过
device: npu:0 run_mode: 0
x shape: (8, 8) scale shape: (8, 8) bias shape: (8, 8)
输出 shape: (8, 8) 最大误差: 0.0
输出前 2x4: tensor([[2.9590, 1.2178, 1.5791, 0.0000],
        [1.0371, 0.0352, 5.1836, 1.7520]], dtype=torch.float16)


`fused_mul_add_relu_kernel` 把三个逐元素步骤串在一起：先做乘法，再加 bias，最后用 `maximum(y, 0.0)` 实现 ReLU。每一步都保持相同 shape，最后通过 `out.move(...)` 写回结果。

### 代码细节解释

逐行看这个 Kernel：

- `pypto.mul(x, scale)`：完成逐元素乘法，输出 shape 仍为 `[8, 8]`。
- `pypto.add(..., bias)`：继续逐元素加 Bias，shape 不变。
- `pypto.maximum(y, 0.0)`：将小于 0 的元素截断为 0，起到 ReLU 的效果。
- `out.move(...)`：将最终结果写回输出 Tensor。

因为所有步骤都是逐元素计算，所以仍然使用 `set_vec_tile_shapes`。如果未来把矩阵乘法加入同一个函数，就需要重新分析是否还涉及 Cube Tile。


这个算子可以按数据流阅读：

```text
x, scale
  -> pypto.mul(x, scale)
  -> pypto.add(..., bias)
  -> pypto.maximum(..., 0.0)
  -> out.move(...)
```

所有步骤都是逐元素计算，所以输出 shape 始终和输入 `x` 一致。验证时，PyTorch 参考实现也按同一个数据流写成 `torch.maximum(x * scale + bias, zero)`。

这个单元用来验证 `fused_mul_add_relu_kernel`。它先在当前设备上构造一组小尺寸输入，再提前分配输出 Tensor 并调用 PyPTO kernel。随后用普通 PyTorch 写出等价的 PyTorch 参考结果，打印 shape、样例值和最大误差；最后的 `assert_close` 是正式检查，误差超出阈值时会直接报错。


**预期输出说明**

运行成功后，会看到 `fused_mul_add_relu_kernel 验证通过`。输出前几项中小于 0 的中间结果会被 ReLU 截断为 0。


## 6. arange 创建序列

`pypto.arange` 用来在 kernel 中创建一段等差序列。覆盖三种调用方式：`arange(end)`、`arange(start, end)` 和 `arange(start, end, step)`。

In [13]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def arange_end_kernel(out: pypto.Tensor((4,), pypto.DT_INT32), end):
    pypto.set_vec_tile_shapes(8)
    out.move(pypto.arange(end))


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def arange_start_end_kernel(out: pypto.Tensor((3,), pypto.DT_FP32), start, end):
    pypto.set_vec_tile_shapes(8)
    out.move(pypto.arange(start, end))


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def arange_start_end_step_kernel(out: pypto.Tensor((6,), pypto.DT_FP32), start, end, step):
    pypto.set_vec_tile_shapes(8)
    out.move(pypto.arange(start, end, step))


def main_arange():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    expected_a = torch.tensor([0, 1, 2, 3], dtype=torch.int32, device=device)
    out_a = torch.empty(4, dtype=torch.int32, device=device)
    arange_end_kernel(out_a, end=4)
    assert_allclose(out_a.cpu().numpy(), expected_a.cpu().numpy(), rtol=1e-3, atol=1e-3)

    expected_b = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32, device=device)
    out_b = torch.empty(3, dtype=torch.float32, device=device)
    arange_start_end_kernel(out_b, start=1.0, end=4.0)
    assert_allclose(out_b.cpu().numpy(), expected_b.cpu().numpy(), rtol=1e-3, atol=1e-3)

    expected_c = torch.tensor([1.0, 1.5, 2.0, 2.5, 3.0, 3.5], dtype=torch.float32, device=device)
    out_c = torch.empty(6, dtype=torch.float32, device=device)
    arange_start_end_step_kernel(out_c, start=1.0, end=4.0, step=0.5)
    assert_allclose(out_c.cpu().numpy(), expected_c.cpu().numpy(), rtol=1e-3, atol=1e-3)

    print("arange 验证通过")
    print("arange(end):", out_a.cpu())
    print("arange(start, end):", out_b.cpu())
    print("arange(start, end, step):", out_c.cpu())


main_arange()


arange 验证通过
arange(end): tensor([0, 1, 2, 3], dtype=torch.int32)
arange(start, end): tensor([1., 2., 3.])
arange(start, end, step): tensor([1.0000, 1.5000, 2.0000, 2.5000, 3.0000, 3.5000])


## 7. full 创建填充值 Tensor

`pypto.full(shape, fill_value, dtype)` 用来创建所有元素都等于同一个值的 Tensor。下面覆盖两个版本：普通 float 填充值，以及整数填充值。


In [14]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def full_float_kernel(out: pypto.Tensor((2, 2), pypto.DT_FP32), fill_value):
    pypto.set_vec_tile_shapes(2, 8)
    out.move(pypto.full((2, 2), fill_value, pypto.DT_FP32))


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def full_int_kernel(out: pypto.Tensor((2, 2), pypto.DT_INT32), fill_value):
    pypto.set_vec_tile_shapes(2, 8)
    out.move(pypto.full((2, 2), fill_value, pypto.DT_INT32))


def main_full():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    expected_a = torch.tensor([[1.0, 1.0], [1.0, 1.0]], dtype=torch.float32, device=device)
    out_a = torch.empty((2, 2), dtype=torch.float32, device=device)
    full_float_kernel(out_a, fill_value=1.0)
    assert_allclose(out_a.cpu().numpy(), expected_a.cpu().numpy(), rtol=1e-3, atol=1e-3)

    expected_b = torch.tensor([[1, 1], [1, 1]], dtype=torch.int32, device=device)
    out_b = torch.empty((2, 2), dtype=torch.int32, device=device)
    full_int_kernel(out_b, fill_value=1)
    assert_allclose(out_b.cpu().numpy(), expected_b.cpu().numpy(), rtol=1e-3, atol=1e-3)

    print("full 验证通过")
    print("float full output:", out_a.cpu())
    print("int full output:", out_b.cpu())


main_full()


full 验证通过
float full output: tensor([[1., 1.],
        [1., 1.]])
int full output: tensor([[1, 1],
        [1, 1]], dtype=torch.int32)


## 8. 逐元素 API 全量覆盖

前面几节已经讲清楚逐元素计算的基本模型。下面按模式归类列出所有 beginner 逐元素 API，每个 kernel 的写法、输入构造和 PyTorch 参考验证都遵循同一结构：设置 vec tile，调用 `pypto.xxx(...)`，写入 `out`，再用 PyTorch 参考结果验证。

In [15]:
def check_close(name, actual, expected, rtol=1e-3, atol=1e-3):
    assert_allclose(actual.cpu().numpy(), expected.cpu().numpy(), rtol=rtol, atol=atol)
    print(f"{name} verified")
    print("  output:", actual.cpu())
    print("  expected:", expected.cpu())

### 8.1 二元计算、标量、广播和 alpha

`add`、`sub`、`mul`、`div` 都遵循同一类模式：两个 Tensor 对应位置计算，也可以和标量计算，shape 不一致时按广播规则对齐。`add` 和 `sub` 还演示了 `alpha` 参数。`clip` 演示三输入逐元素限幅。

In [16]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def api_add_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.add(a, b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def api_add_broadcast_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.add(a, b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def api_add_scalar_kernel(x: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32), scalar: float):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.add(x, scalar)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def api_add_with_alpha_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32), alpha: float):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.add(a, b, alpha=alpha)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def sub_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.sub(a, b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def sub_broadcast_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.sub(a, b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def sub_scalar_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32), scalar: float):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.sub(a, scalar)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def sub_with_alpha_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32), alpha: float):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.sub(a, b, alpha=alpha)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def mul_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.mul(a, b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def mul_broadcast_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.mul(a, b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def mul_scalar_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32), scalar: float):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.mul(a, scalar)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def div_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.div(a, b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def div_broadcast_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.div(a, b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def div_scalar_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32), scalar: float):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.div(a, scalar)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def clip_kernel(a: pypto.Tensor([], pypto.DT_FP32), min_: pypto.Tensor([], pypto.DT_FP32), max_: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.clip(a, min_, max_)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def clip_broadcast_kernel(a: pypto.Tensor([], pypto.DT_FP32), min_: pypto.Tensor([], pypto.DT_FP32), max_: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.clip(a, min_, max_)


def test_binary_elementwise_examples():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    dtype = torch.float32

    a = torch.tensor([[1, 2, 3]], dtype=dtype, device=device)
    b = torch.tensor([[4, 5, 6]], dtype=dtype, device=device)
    out = torch.empty_like(a)
    api_add_kernel(a, b, out)
    check_close("add::test_add_basic", out, a + b)

    a2 = torch.tensor([[1, 2], [3, 4]], dtype=dtype, device=device)
    b2 = torch.tensor([[1, 2]], dtype=dtype, device=device)
    out = torch.empty_like(a2)
    api_add_broadcast_kernel(a2, b2, out)
    check_close("add::test_add_broadcast", out, a2 + b2)

    out = torch.empty_like(a)
    api_add_scalar_kernel(a, out, 2.0)
    check_close("add::test_add_scalar", out, a + 2.0)

    out = torch.empty_like(a)
    api_add_with_alpha_kernel(a, b, out, 2.0)
    check_close("add::test_add_with_alpha", out, a + 2.0 * b)

    a = torch.tensor([[4, 5, 6]], dtype=dtype, device=device)
    b = torch.tensor([[1, 2, 3]], dtype=dtype, device=device)
    out = torch.empty_like(a)
    sub_kernel(a, b, out)
    check_close("sub::test_sub_basic", out, a - b)

    out = torch.empty_like(a2)
    sub_broadcast_kernel(a2, b2, out)
    check_close("sub::test_sub_broadcast", out, a2 - b2)

    out = torch.empty_like(b)
    sub_scalar_kernel(b, out, 2.0)
    check_close("sub::test_sub_scalar", out, b - 2.0)

    a = torch.tensor([[9, 8, 7]], dtype=dtype, device=device)
    b = torch.tensor([[1, 2, 3]], dtype=dtype, device=device)
    out = torch.empty_like(a)
    sub_with_alpha_kernel(a, b, out, 2.0)
    check_close("sub::test_sub_with_alpha", out, a - 2.0 * b)

    a = torch.tensor([[1, 2, 3]], dtype=dtype, device=device)
    b = torch.tensor([[4, 5, 6]], dtype=dtype, device=device)
    out = torch.empty_like(a)
    mul_kernel(a, b, out)
    check_close("mul::test_mul_basic", out, a * b)

    out = torch.empty_like(a2)
    mul_broadcast_kernel(a2, b2, out)
    check_close("mul::test_mul_broadcast", out, a2 * b2)

    out = torch.empty_like(a)
    mul_scalar_kernel(a, out, 2.0)
    check_close("mul::test_mul_scalar", out, a * 2.0)

    a = torch.tensor([[6, 10, 15]], dtype=dtype, device=device)
    b = torch.tensor([[2, 5, 3]], dtype=dtype, device=device)
    out = torch.empty_like(a)
    div_kernel(a, b, out)
    check_close("div::test_div_basic", out, a / b)

    out = torch.empty_like(a2)
    div_broadcast_kernel(a2, b2, out)
    check_close("div::test_div_broadcast", out, a2 / b2)

    a = torch.tensor([[1, 2, 3]], dtype=dtype, device=device)
    out = torch.empty_like(a)
    div_scalar_kernel(a, out, 2.0)
    check_close("div::test_div_scalar", out, a / 2.0)

    x = torch.tensor([[0, 2, 4], [3, 4, 6]], dtype=dtype, device=device)
    min_full = torch.tensor([[1, 1, 1], [1, 1, 1]], dtype=dtype, device=device)
    max_full = torch.tensor([[3, 3, 3], [3, 3, 3]], dtype=dtype, device=device)
    out = torch.empty_like(x)
    clip_kernel(x, min_full, max_full, out)
    check_close("clip::test_clip_basic", out, torch.clamp(x, min_full, max_full))

    min_row = torch.tensor([1, 1, 1], dtype=dtype, device=device)
    max_row = torch.tensor([3, 3, 3], dtype=dtype, device=device)
    out = torch.empty_like(x)
    clip_broadcast_kernel(x, min_row, max_row, out)
    check_close("clip::test_clip_broadcast", out, torch.clamp(x, min_row, max_row))


test_binary_elementwise_examples()


add::test_add_basic verified
  output: tensor([[5., 7., 9.]])
  expected: tensor([[5., 7., 9.]])
add::test_add_broadcast verified
  output: tensor([[2., 4.],
        [4., 6.]])
  expected: tensor([[2., 4.],
        [4., 6.]])
add::test_add_scalar verified
  output: tensor([[3., 4., 5.]])
  expected: tensor([[3., 4., 5.]])
add::test_add_with_alpha verified
  output: tensor([[ 9., 12., 15.]])
  expected: tensor([[ 9., 12., 15.]])
sub::test_sub_basic verified
  output: tensor([[3., 3., 3.]])
  expected: tensor([[3., 3., 3.]])
sub::test_sub_broadcast verified
  output: tensor([[0., 0.],
        [2., 2.]])
  expected: tensor([[0., 0.],
        [2., 2.]])
sub::test_sub_scalar verified
  output: tensor([[-1.,  0.,  1.]])
  expected: tensor([[-1.,  0.,  1.]])
sub::test_sub_with_alpha verified
  output: tensor([[7., 4., 1.]])
  expected: tensor([[7., 4., 1.]])
mul::test_mul_basic verified
  output: tensor([[ 4., 10., 18.]])
  expected: tensor([[ 4., 10., 18.]])
mul::test_mul_broadcast verified


### 8.2 单输入数学函数、幂运算和取整

这一组示例覆盖 `abs`、`exp`、`exp2`、`expm1`、`log`、`neg`、`pow`、`round`、`rsqrt`、`ceil`、`floor`、`trunc` 和 `sqrt`。除 `pow` 外，大多数都是一个输入 Tensor 得到一个同 shape 输出 Tensor。

In [17]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def abs_kernel(x: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.abs(x)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def exp_kernel(x: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.exp(x)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def exp2_kernel(x: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.exp2(x)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def expm1_kernel(x: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.expm1(x)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def log_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.log(a)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def neg_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.neg(a)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def pow_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.pow(a, b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def round_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32), decimals: int):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.round(a, decimals=decimals)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def rsqrt_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.rsqrt(a)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def ceil_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.ceil(a)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def floor_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.floor(a)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def trunc_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.trunc(a)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def sqrt_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.sqrt(a)


def test_unary_elementwise_examples():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    dtype = torch.float32

    x = torch.tensor([[-1, -8, 2]], dtype=dtype, device=device)
    out = torch.empty_like(x)
    abs_kernel(x, out)
    check_close("abs::test_abs_basic", out, torch.abs(x))

    x = torch.tensor([[0, 1, 2]], dtype=dtype, device=device)
    out = torch.empty_like(x)
    exp_kernel(x, out)
    check_close("exp::test_exp_basic", out, torch.exp(x))

    out = torch.empty_like(x)
    exp2_kernel(x, out)
    check_close("exp2::test_exp2_basic", out, torch.exp2(x))

    out = torch.empty_like(x)
    expm1_kernel(x, out)
    check_close("expm1::test_expm1_basic", out, torch.expm1(x))

    x = torch.tensor([[1, 2, 3]], dtype=dtype, device=device)
    out = torch.empty_like(x)
    log_kernel(x, out)
    check_close("log::test_log_basic", out, torch.log(x))

    x = torch.tensor([[1, -2, 3]], dtype=dtype, device=device)
    out = torch.empty_like(x)
    neg_kernel(x, out)
    check_close("neg::test_neg_basic", out, -x)

    a = torch.tensor([[2, 3, 4]], dtype=dtype, device=device)
    b = torch.tensor([[3, 2, 1]], dtype=dtype, device=device)
    out = torch.empty_like(a)
    pow_kernel(a, b, out)
    check_close("pow::test_pow_basic", out, torch.pow(a, b))

    x = torch.tensor([[1.234, 4.567], [16.891, 9.345]], dtype=dtype, device=device)
    out = torch.empty_like(x)
    round_kernel(x, out, 1)
    check_close("round::test_round_basic", out, torch.round(x, decimals=1))

    x = torch.tensor([[1, 4], [16, 9]], dtype=dtype, device=device)
    out = torch.empty_like(x)
    rsqrt_kernel(x, out)
    check_close("rsqrt::test_rsqrt_basic", out, torch.rsqrt(x))

    x = torch.tensor([[1.2, 4.7], [-1.1, 9.0]], dtype=dtype, device=device)
    out = torch.empty_like(x)
    ceil_kernel(x, out)
    check_close("ceil::test_ceil_basic", out, torch.ceil(x))

    out = torch.empty_like(x)
    floor_kernel(x, out)
    check_close("floor::test_floor_basic", out, torch.floor(x))

    x = torch.tensor([[1.2, 4.1], [16.8, 9.3]], dtype=dtype, device=device)
    out = torch.empty_like(x)
    trunc_kernel(x, out)
    check_close("trunc::test_trunc_basic", out, torch.trunc(x))

    x = torch.tensor([[1, 4], [16, 9]], dtype=dtype, device=device)
    out = torch.empty_like(x)
    sqrt_kernel(x, out)
    check_close("sqrt::test_sqrt_basic", out, torch.sqrt(x))


test_unary_elementwise_examples()


abs::test_abs_basic verified
  output: tensor([[1., 8., 2.]])
  expected: tensor([[1., 8., 2.]])
exp::test_exp_basic verified
  output: tensor([[1.0000, 2.7183, 7.3891]])
  expected: tensor([[1.0000, 2.7183, 7.3891]])
exp2::test_exp2_basic verified
  output: tensor([[1., 2., 4.]])
  expected: tensor([[1., 2., 4.]])
expm1::test_expm1_basic verified
  output: tensor([[0.0000, 1.7183, 6.3891]])
  expected: tensor([[0.0000, 1.7183, 6.3891]])
log::test_log_basic verified
  output: tensor([[0.0000, 0.6931, 1.0986]])
  expected: tensor([[0.0000, 0.6931, 1.0986]])
neg::test_neg_basic verified
  output: tensor([[-1.,  2., -3.]])
  expected: tensor([[-1.,  2., -3.]])
pow::test_pow_basic verified
  output: tensor([[8., 9., 4.]])
  expected: tensor([[8., 9., 4.]])
round::test_round_basic verified
  output: tensor([[ 1.2000,  4.6000],
        [16.9000,  9.3000]])
  expected: tensor([[ 1.2000,  4.6000],
        [16.9000,  9.3000]])
rsqrt::test_rsqrt_basic verified
  output: tensor([[1.0000, 0.5000],

### 8.3 逐元素 API 小结

本节把 `elementwise_ops.py` 中的逐元素 API 示例完整搬到教程侧，并按模式归类：二元计算、广播、标量、`alpha`、限幅、单输入数学函数、幂运算和取整。

这些 API 的阅读方法是一致的：先看输入和输出 shape，再看是否涉及广播或标量，最后用 PyTorch 参考实现确认数值语义。

## 9. 课后练习

练习目标是在融合乘加 ReLU 的基础上继续增加平方操作，使算子完成：

```python
y = maximum(x * scale + bias, 0)
out = y * y
```

这个练习仍然是逐元素计算。新增的平方操作不会改变 shape，只是在 ReLU 结果的每个位置上再乘一次自身。

建议步骤：

1. 复制 `fused_mul_add_relu_kernel` 的数据流。
2. 在 `maximum` 后增加一次逐元素乘法。
3. 使用 PyTorch 参考实现验证。

参考答案：

```python
relu = pypto.maximum(y, 0.0)
out.move(pypto.mul(relu, relu))
```


### 可运行课后练习：增加平方输出

下面给出可直接验证的实现：

```python
relu = maximum(x * scale + bias, 0)
out = relu * relu
```

这个练习继续使用向量 Tile，因为所有计算都发生在对应元素之间。


In [ ]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def fused_square_practice_kernel(
    x: pypto.Tensor([], pypto.DT_FP16),
    scale: pypto.Tensor([], pypto.DT_FP16),
    bias: pypto.Tensor([], pypto.DT_FP16),
    out: pypto.Tensor([], pypto.DT_FP16)):
    pypto.set_vec_tile_shapes(8, 8)
    y = pypto.add(pypto.mul(x, scale), bias)
    relu = pypto.maximum(y, 0.0)
    squared = pypto.mul(relu, relu)
    out.move(squared)


def main_fused_square_practice():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.randn((8, 8), dtype=torch.float16, device=device)
    scale = torch.randn((8, 8), dtype=torch.float16, device=device)
    bias = torch.randn((8, 8), dtype=torch.float16, device=device)
    out = torch.empty_like(x)

    fused_square_practice_kernel(x, scale, bias, out)
    zero = torch.tensor(0.0, dtype=torch.float16, device=device)
    ref = torch.maximum(x * scale + bias, zero)
    ref = ref * ref

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-2, atol=1e-2)
    print("fused_square_practice_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("输出 shape:", tuple(out.shape), "最大误差:", max_diff)
    print("输出前 2x4:", out[:2, :4].cpu())


main_fused_square_practice()


这个模块在融合乘加和 ReLU 的基础上继续增加一步逐元素平方。`relu` 与自身相乘得到 `squared`，最后通过 `out.move(squared)` 写回输出。整个计算链路的 shape 始终保持 `[8, 8]`。


验证逻辑使用 PyTorch 写出同样的数据流：先计算 `torch.maximum(x * scale + bias, 0)`，再做 `ref * ref`。两边最大误差在阈值内时，说明融合写法与参考实现一致。


**预期输出说明**

运行成功后，会看到 `fused_square_practice_kernel 验证通过`，并打印当前 device、run_mode、输出 shape、最大误差和输出前几项。


核心语句是：

```python
squared = pypto.mul(relu, relu)
out.move(squared)
```


## 10. 本节小结

本节从 shape 最直观的逐元素计算开始，依次解释了标量、广播、vec tile、输出写回和验证闭环。补充部分覆盖了 kernel 内 Tensor 创建（arange、full）和逐元素 API 全量清单。

需要重点记住：逐元素算子的核心是各位置独立计算；广播解决 shape 对齐问题；`set_vec_tile_shapes` 组织执行但不改变数学结果；PyTorch 参考实现用于证明 PyPTO kernel 的输出正确。